In [40]:
import os
import time
import sqlparse
import warnings
import pymysql
import numpy as np
import pandas as pd
import datetime as dt
import haversine as hs
from tqdm import tqdm, trange
from dotenv import load_dotenv

# Testing Queries for Seiscomp6

In previous versions of the revision routine, there are a fixed set of queries that are used to test the connection to the database and to extract the info needed to run the revision. Here, we test the same queries for Seiscomp6, and make sure they work as expected. We will also test the connection to the database, and make sure that we can ping the server. In addition, by using DBeaver I try to extend the queries to extract more info that may be useful for the revision, mainly for those events with 6 to 8 phases.

Let's creating a single function that queries some info from the seiscomp database.

In [41]:
env_path = os.path.join(os.getcwd(), '.env')
load_dotenv(dotenv_path=env_path)

def connect_to_db(
        query: str,
        start_time: dt.datetime = None,
        end_time: dt.datetime = None,
        **kwargs):

    if start_time and end_time:
        start_time_str = start_time.strftime("%Y-%m-%d %H:%M:%S")
        end_time_str = end_time.strftime("%Y-%m-%d %H:%M:%S")
        full_query = f"{query} '{start_time_str}' and '{end_time_str}' ORDER BY Origin.time_value ASC;"  # Filter and order by time_value
    else:
        full_query = query

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        db_connection = pymysql.connect(
            host=os.getenv('SERVER_HOST'),
            user=os.getenv('SERVER_USERNAME'),
            password=os.getenv('SERVER_PASSWORD'),
            db=os.getenv('SERVER_DATABASE')
        )

        try:
            with tqdm(total=1, desc='Querying database...', unit='query', leave=False,bar_format="{desc}") as pbar:
                df = pd.read_sql_query(full_query, db_connection, **kwargs)
                pbar.update(1)
        finally:
            db_connection.close()

    return df

# Unboxing current queries

Let's start by unboxing the current queries used in the revision routine, and test them one by one.

## Normal queries

The SQL query used is:

```sql
Select Origin.time_value, POEv.publicID, Origin.depth_value, Magnitude.magnitude_value, Origin.quality_standardError, Origin.depth_uncertainty, Origin.latitude_uncertainty, Origin.longitude_uncertainty, Origin.quality_associatedPhaseCount, Origin.creationInfo_author, Event.type, Origin.creationInfo_agencyID, EventDescription.text, Origin.latitude_value, Origin.longitude_value, Magnitude.type, Origin.methodID, Origin.earthModelID from Event AS EvMF left join PublicObject AS POEv ON EvMF._oid = POEv._oid left join PublicObject as POOri ON EvMF.preferredOriginID=POOri.publicID left join Origin ON POOri._oid=Origin._oid left join PublicObject as POMag on EvMF.preferredMagnitudeID=POMag.publicID left join Magnitude ON Magnitude._oid = POMag._oid left join Event ON Event._oid= POEv._oid left join EventDescription ON EvMF._oid = EventDescription._parent_oid where Origin.time_value between
```

where:

- `EvMF` is the Event table, which contains the main information about the event, such as the origin time, depth, magnitude, etc.
- `POEv` is the PublicObject table, which contains the public ID of the event, which is used to link the Event table with the Origin and Magnitude tables.
- `POOri` is the PublicObject table, which contains the public ID of the origin, which is used to link the Origin table with the Event table.
- `Origin` is the Origin table, which contains the information about the origin of the event, such as the time, depth, latitude, longitude, etc.
- `POMag` is the PublicObject table, which contains the public ID of the magnitude, which is used to link the Magnitude table with the Event table.
- `Magnitude` is the Magnitude table, which contains the information about the magnitude of the event, such as the magnitude value, type, etc.
- `EventDescription` is the EventDescription table, which contains the description of the event, such as the text description, etc.

In this SQL query, we are selecting the following fields:
- `Origin.time_value`: the origin time of the event
- `POEv.publicID`: the public ID of the event
- `Origin.depth_value`: the depth of the event
- `Magnitude.magnitude_value`: the magnitude of the event
- `Origin.quality_standardError`: the standard error of the origin
- `Origin.depth_uncertainty`: the uncertainty of the depth
- `Origin.latitude_uncertainty`: the uncertainty of the latitude
- `Origin.longitude_uncertainty`: the uncertainty of the longitude
- `Origin.quality_associatedPhaseCount`: the number of associated phases
- `Origin.creationInfo_author`: the author of the origin
- `Event.type`: the type of the event
- `Origin.creationInfo_agencyID`: the agency ID of the origin
- `EventDescription.text`: the text description of the event
- `Origin.latitude_value`: the latitude of the event
- `Origin.longitude_value`: the longitude of the event
- `Magnitude.type`: the type of the magnitude
- `Origin.methodID`: the method ID of the origin
- `Origin.earthModelID`: the earth model ID of the origin

and the condition is that the origin time of the event is between a certain time range, which is specified in the last part of the query.

Let's test this query directly in this notebook, by using the function from the last cell. We will see as a simple example the query for a specific time range, to see if we can retrieve all the origins that have been created between a time range.

In [42]:
query_example = "SELECT * FROM Origin WHERE Origin.time_value BETWEEN"
origin_df = connect_to_db(query_example, start_time=dt.datetime(2026, 5, 1), end_time=dt.datetime.now(dt.UTC))
origin_df

,_oid,_parent_oid,_last_modified,time_value,time_value_ms,time_uncertainty,time_lowerUncertainty,time_upperUncertainty,time_confidenceLevel,time_pdf_variable_content,...,creationInfo_agencyID,creationInfo_agencyURI,creationInfo_author,creationInfo_authorURI,creationInfo_creationTime,creationInfo_creationTime_ms,creationInfo_modificationTime,creationInfo_modificationTime_ms,creationInfo_version,creationInfo_used
0,736936910,1,2026-05-01 10:10:32,2026-05-01 00:11:51,300579,NaN,None,None,None,None,...,SGC,,muruena@proc3,,2026-05-01 02:39:59,370508,2026-05-01 02:40:12,540503.0,,1
1,736933980,1,2026-05-01 10:10:24,2026-05-01 00:11:51,574909,0.498633,None,None,None,None,...,SGC,,scanloc,,2026-05-01 00:12:30,606635,NaT,NaN,,1
2,736933899,1,2026-05-01 10:10:23,2026-05-01 00:11:51,688749,0.639198,None,None,None,None,...,SGC,,scanloc,,2026-05-01 00:12:25,749642,NaT,NaN,,1
3,736934801,1,2026-05-01 10:10:26,2026-05-01 00:11:51,890000,NaN,None,None,None,None,...,SGC,,gerard@proc3,,2026-05-01 00:14:08,106973,2026-05-01 00:14:53,486555.0,,1
4,736934096,1,2026-05-01 10:10:24,2026-05-01 00:11:51,921698,0.370669,None,None,None,None,...,SGC,,scanloc,,2026-05-01 00:12:37,435980,NaT,NaN,,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3352,766827757,1,2026-06-06 23:05:20,2026-06-06 22:07:10,106606,0.615602,None,None,None,None,...,SGC,,scanloc,,2026-06-06 22:08:03,229277,NaT,NaN,,1
3353,766829104,1,2026-06-06 23:05:24,2026-06-06 22:07:10,480340,0.420609,None,None,None,None,...,SGC,,scanloc,,2026-06-06 22:11:17,43072,NaT,NaN,,1
3354,766828145,1,2026-06-06 23:05:21,2026-06-06 22:07:10,645219,0.457839,None,None,None,None,...,SGC,,scanloc,,2026-06-06 22:08:26,559293,NaT,NaN,,1
3355,766828259,1,2026-06-06 23:05:21,2026-06-06 22:07:11,771339,0.446576,None,None,None,None,...,SGC,,scanloc,,2026-06-06 22:08:32,77247,NaT,NaN,,1


To check, as another example, the info from all preferred origins between two time ranges, we can use the same query stated earlier:

In [43]:
initial_time = dt.datetime(2023, 5, 3, 0, 0, 0)
final_time = dt.datetime(2026, 3, 17, 0, 0, 0)

In [44]:
query2 = "Select Origin.time_value, POEv.publicID, Origin.depth_value, Magnitude.magnitude_value, Origin.quality_standardError, Origin.depth_uncertainty, Origin.latitude_uncertainty, Origin.longitude_uncertainty, Origin.quality_associatedPhaseCount, Origin.creationInfo_author, Event.type, Origin.creationInfo_agencyID, EventDescription.text, Origin.latitude_value, Origin.longitude_value, Magnitude.type, Origin.methodID, Origin.earthModelID from Event AS EvMF left join PublicObject AS POEv ON EvMF._oid = POEv._oid left join PublicObject as POOri ON EvMF.preferredOriginID=POOri.publicID left join Origin ON POOri._oid=Origin._oid left join PublicObject as POMag on EvMF.preferredMagnitudeID=POMag.publicID left join Magnitude ON Magnitude._oid = POMag._oid left join Event ON Event._oid= POEv._oid left join EventDescription ON EvMF._oid = EventDescription._parent_oid where Origin.time_value between"
event_df2 = connect_to_db(query2, start_time=initial_time, end_time=final_time)
event_df2

,time_value,publicID,depth_value,magnitude_value,quality_standardError,depth_uncertainty,latitude_uncertainty,longitude_uncertainty,quality_associatedPhaseCount,creationInfo_author,type,creationInfo_agencyID,text,latitude_value,longitude_value,type,methodID,earthModelID
0,2023-05-03 00:16:51,SGC2023ipzdxf,10.000000,NaN,10.420535,NaN,NaN,NaN,NaN,eguzman@proc3,not locatable,SGC,"CalarcÃ¡ - QuindÃ­o, Colombia",4.484200,-75.636700,None,,
1,2023-05-03 00:22:16,SGC2023ipzion,-4.941406,0.885335,0.445758,5.790340,2.080092,5.282553,8.0,eguzman@proc3,not locatable,SGC,"Colombia-Ecuador, Region Fronteriza",0.781972,-77.914268,MLr,NonLinLoc,Poveda_et_al_2018
2,2023-05-03 01:17:28,SGC2023iqbedm,135.859375,1.911117,0.959775,9.021321,6.419839,7.434003,19.0,eguzman@proc3,earthquake,SGC,"Zapatoca - Santander, Colombia",6.766267,-73.218474,MLr_3,NonLinLoc,Poveda_et_al_2018
3,2023-05-03 01:24:17,SGC2023iqbkaq,22.539062,0.880860,0.356193,7.230936,5.458100,3.798461,12.0,eguzman@proc3,earthquake,SGC,"PurificaciÃ³n - Tolima, Colombia",3.844233,-74.806306,MLr_2,NonLinLoc,Poveda_et_al_2018
4,2023-05-03 01:28:27,SGC2023iqbnpk,10.000000,NaN,3.682291,NaN,NaN,NaN,NaN,eguzman@proc3,not locatable,SGC,Mar Caribe,8.644500,-77.360400,None,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
209238,2026-03-16 23:11:29,SGC2026fhpwfl,136.328125,2.074670,0.956817,7.253506,5.312920,6.076712,28.0,hmoreno@proc2,earthquake,SGC,"CÃ¡chira - Norte de Santander, Colombia",7.756859,-73.141695,MLr_3,NonLinLoc,Poveda_et_al_2018
209239,2026-03-16 23:18:05,SGC2026fhqbxv,103.632812,1.727774,0.597553,8.010460,4.339104,4.863137,22.0,hmoreno@proc2,earthquake,SGC,"Pauna - BoyacÃ¡, Colombia",5.615868,-73.956240,MLr_3,NonLinLoc,Poveda_et_al_2018
209240,2026-03-16 23:31:36,SGC2026fhqnnx,10.000000,-0.143127,13.207386,NaN,NaN,NaN,NaN,hmoreno@proc2,not locatable,SGC,"Dabeiba - Antioquia, Colombia",7.018000,-76.210000,MLr_1,,
209241,2026-03-16 23:35:30,SGC2026fhqqxh,10.000000,-0.012095,223.843471,NaN,NaN,NaN,NaN,hmoreno@proc2,not locatable,SGC,"Dabeiba - Antioquia, Colombia",7.018000,-76.210000,MLr_1,,


However, it is more convenient to read the query from a .sql file, to avoid having a very long query in the notebook. We can read the query from a .sql file, and then use it in the function. Let's do it for the previous query, but adding the following columns:

1. The used stations for the localization.
2. The total stations that have associated phases.
3. The used phases for the localization.
4. The Comments section used to define a DESTACADO event.

In [45]:
# Read revision.sql file
with open('./queries/revision.sql', 'r') as file:
    revision_query = file.read()

clean_sql = sqlparse.format(revision_query, strip_comments=True).strip()

In [46]:
event_df3 = connect_to_db(clean_sql, start_time=initial_time, end_time=final_time)
event_df3

,time_value,publicID,depth_value,magnitude_value,quality_standardError,depth_uncertainty,latitude_uncertainty,longitude_uncertainty,quality_associatedPhaseCount,quality_usedPhaseCount,...,quality_associatedStationCount,event_type,creationInfo_agencyID,text,latitude_value,longitude_value,magnitude_type,methodID,earthModelID,comment
0,2023-05-03 00:16:51,SGC2023ipzdxf,10.000000,NaN,10.420535,NaN,NaN,NaN,NaN,NaN,...,NaN,not locatable,SGC,"CalarcÃ¡ - QuindÃ­o, Colombia",4.484200,-75.636700,None,,,None
1,2023-05-03 00:22:16,SGC2023ipzion,-4.941406,0.885335,0.445758,5.790340,2.080092,5.282553,8.0,8.0,...,NaN,not locatable,SGC,"Colombia-Ecuador, Region Fronteriza",0.781972,-77.914268,MLr,NonLinLoc,Poveda_et_al_2018,None
2,2023-05-03 01:17:28,SGC2023iqbedm,135.859375,1.911117,0.959775,9.021321,6.419839,7.434003,19.0,19.0,...,NaN,earthquake,SGC,"Zapatoca - Santander, Colombia",6.766267,-73.218474,MLr_3,NonLinLoc,Poveda_et_al_2018,None
3,2023-05-03 01:24:17,SGC2023iqbkaq,22.539062,0.880860,0.356193,7.230936,5.458100,3.798461,12.0,12.0,...,NaN,earthquake,SGC,"PurificaciÃ³n - Tolima, Colombia",3.844233,-74.806306,MLr_2,NonLinLoc,Poveda_et_al_2018,None
4,2023-05-03 01:28:27,SGC2023iqbnpk,10.000000,NaN,3.682291,NaN,NaN,NaN,NaN,NaN,...,NaN,not locatable,SGC,Mar Caribe,8.644500,-77.360400,None,,,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
209238,2026-03-16 23:11:29,SGC2026fhpwfl,136.328125,2.074670,0.956817,7.253506,5.312920,6.076712,28.0,27.0,...,NaN,earthquake,SGC,"CÃ¡chira - Norte de Santander, Colombia",7.756859,-73.141695,MLr_3,NonLinLoc,Poveda_et_al_2018,None
209239,2026-03-16 23:18:05,SGC2026fhqbxv,103.632812,1.727774,0.597553,8.010460,4.339104,4.863137,22.0,22.0,...,NaN,earthquake,SGC,"Pauna - BoyacÃ¡, Colombia",5.615868,-73.956240,MLr_3,NonLinLoc,Poveda_et_al_2018,None
209240,2026-03-16 23:31:36,SGC2026fhqnnx,10.000000,-0.143127,13.207386,NaN,NaN,NaN,NaN,NaN,...,NaN,not locatable,SGC,"Dabeiba - Antioquia, Colombia",7.018000,-76.210000,MLr_1,,,None
209241,2026-03-16 23:35:30,SGC2026fhqqxh,10.000000,-0.012095,223.843471,NaN,NaN,NaN,NaN,NaN,...,NaN,not locatable,SGC,"Dabeiba - Antioquia, Colombia",7.018000,-76.210000,MLr_1,,,None


By using this new SQL query, it is not required to use the previous two querys to select DESTACADO and normal events, since all the info is contained in the same query.

# Optimizing Duplicates Search

Actually, the revision routine includes a function to check for duplicates, which is based on the time and geographical distance between events. However, this function is not optimized, since it loops over all the events and compares them one by one. This can be very time-consuming, especially if there are many events in the database. The goal of this section is to optimize the duplicates search, by using a more efficient algorithm that can reduce the time complexity of the search. Let's start by defining the previous function to check duplicates:

In [47]:
columns = ['time_value', 'publicID', 'text', 'depth_value', 'magnitude_value', 'magnitude_type',
           'quality_standardError', 'depth_uncertainty', 'latitude_uncertainty', 'longitude_uncertainty',
           'quality_associatedPhaseCount', 'creationInfo_author', 'event_type', 'creationInfo_agencyID']

def check_duplicates(
        events: pd.DataFrame
):
    """
    Identifies duplicate events based on time and geographical distance.

    Parameters:
    -----------
    events: pandas dataframe
        Seismic data for the given time range. Generally obtained from the connect2mysql function.

    Returns:
    --------
    duplicates: A pandas dataframe
        Table with information about duplicate events.
    """
    # Filter events with the following types
    checks = ["earthquake", "volcanic eruption", "explosion", "outside of network interest"]
    selections = events[events['event_type'].isin(checks)].reset_index(drop=True)

    # Make a pandas dataframe to store the info of the duplicates
    duplicates = pd.DataFrame()

    # Loop over al earthquakes, ordered by time
    for i in trange(len(selections) - 1, desc="Checking duplicates", unit=" events", leave=False):
        event1 = selections.iloc[i]  # Take the i-esim event on the list
        event2 = selections.iloc[i + 1]  # Compared to the next event

        # Check if the events are within 4 seconds of each other
        time_diff = abs((event2['time_value'] - event1['time_value']).total_seconds())
        if time_diff <= 4:
            # Estimate the distance between the two events using haversine formula
            distance = hs.haversine((event1['latitude_value'], event1['longitude_value']),
                                     (event2['latitude_value'], event2['longitude_value']))
            if distance <= 100:  # If both events are within 100 km and 4 seconds, they can be duplicates
                dup_1 = event1[columns].copy()
                dup_1['Observations'] = f'Possible duplicate event of {event2["publicID"]}'
                dup_2 = event2[columns].copy()
                dup_2['Observations'] = f'Possible duplicate event of {event1["publicID"]}'
                # Add the two events to the duplicates dataframe
                duplicates = pd.concat([duplicates, dup_1.to_frame().T], ignore_index=True)
                duplicates = pd.concat([duplicates, dup_2.to_frame().T], ignore_index=True)
    return duplicates

As you can see from the function, there are two criteria to consider an event as a duplicate: the time difference between the two events must be less than or equal to 4 seconds, and the geographical distance between the two events must be less than or equal to 100 km. The function loops over all the events, and compares each event with the next one in the list. If both criteria are met, the two events are considered duplicates, and their information is stored in a new dataframe called duplicates.

The idea of checking only the next event in the list is based on the fact that usually the time difference between earthquakes is greater than 4 seconds, so it is unlikely that two events that are far apart in the list are duplicates. Let's test the results for all the 87327 earthquakes from 2023-05-03 to 2026-03-17 (the end of seiscomp3 db), and see how many duplicates we can find and the time it takes to run the function.

In [48]:
time1 = time.time()
duplicates_previous = check_duplicates(event_df3)
time2 = time.time()
print(f"Number of duplicates found: {len(duplicates_previous)}")
print(f"Time taken to run the function: {time2 - time1} seconds")

Number of duplicates found: 96
Time taken to run the function: 16.74306583404541 seconds


## Micro-optimization of the adjacent-only logic

The previous function is based on the adjacent-only logic, which means that it only compares each event with the next one in the list. This is a good approach to reduce the time complexity of the search, but it can be further optimized by using some micro-optimizations. For example, we can pre-extract the relevant columns as NumPy arrays to avoid the overhead of pandas indexing in the loop. We can also accumulate the duplicate rows in a list and create the DataFrame at the end, instead of concatenating in each iteration. Let's implement this micro-optimized version of the adjacent-only logic:

In [49]:
def check_duplicates_adjacent(events: pd.DataFrame) -> pd.DataFrame:
    """
    Identifies duplicate events based on time and geographical distance.
    Same adjacent-only logic as the original, but micro-optimized:
      - Pre-extracted NumPy arrays (avoids per-row iloc overhead)
      - Row accumulation in a list (avoids repeated pd.concat in loop)
      - Single pd.concat at the end

    Time complexity: O(n) comparisons — only i vs i+1
    Parameters:
    -----------
    events : pd.DataFrame
        Seismic data for the given time range.
    Returns:
    --------
    duplicates : pd.DataFrame
        Table with information about duplicate events.
    """
    checks = ["earthquake", "volcanic eruption", "explosion", "outside of network interest"]
    selections = events[events['event_type'].isin(checks)].reset_index(drop=True)

    # Edge case: empty or single-event dataset
    if len(selections) < 2:
        return pd.DataFrame(columns=columns + ['Observations'])

    # --- Pre-extract arrays for fast access ---
    times  = selections['time_value'].values          # numpy datetime64
    lats   = selections['latitude_value'].values.astype(np.float64)
    lons   = selections['longitude_value'].values.astype(np.float64)
    ids    = selections['publicID'].values

    TIME_WINDOW    = np.timedelta64(4, 's')
    DIST_THRESHOLD = 100  # km

    dup_rows = []  # Accumulate rows here, build DF once at the end

    for i in trange(len(selections) - 1, desc="Checking duplicates", unit=" events", leave=False):
        # Time check using numpy (faster than .total_seconds() on Timedelta)
        if abs(times[i + 1] - times[i]) <= TIME_WINDOW:
            distance = hs.haversine((lats[i], lons[i]), (lats[i + 1], lons[i + 1]))
            if distance <= DIST_THRESHOLD:
                row_i = selections.iloc[i][columns].copy()
                row_i['Observations'] = f'Possible duplicate event of {ids[i + 1]}'
                row_j = selections.iloc[i + 1][columns].copy()
                row_j['Observations'] = f'Possible duplicate event of {ids[i]}'
                dup_rows.extend([row_i, row_j])

    # Concat duplicates to one single Dataframe
    if dup_rows:
        duplicates = pd.DataFrame(dup_rows, columns=columns + ['Observations']).reset_index(drop=True)
    else:
        duplicates = pd.DataFrame(columns=columns + ['Observations'])

    return duplicates

In [50]:
# Repeat the test with the adjacent-only logic, but micro-optimized
time1 = time.time()
duplicates_adjacent = check_duplicates_adjacent(event_df3)
time2 = time.time()
print(f"Number of duplicates found: {len(duplicates_adjacent)}")
print(f"Time taken to run the function: {time2 - time1} seconds")

Number of duplicates found: 96
Time taken to run the function: 0.8999950885772705 seconds


In [51]:
# Check if the results are the same
print(f"Number of duplicates found with original adjacent-only: {len(duplicates_previous)}")
print(f"Number of duplicates found with micro-optimized adjacent-only: {len(duplicates_adjacent)}")

Number of duplicates found with original adjacent-only: 96
Number of duplicates found with micro-optimized adjacent-only: 96


In [52]:
# Find differences between the two results
differences = pd.concat([duplicates_previous, duplicates_adjacent, duplicates_adjacent]).drop_duplicates(keep=False)
print(f"Number of differences between original and micro-optimized adjacent-only: {len(differences)}")
differences

Number of differences between original and micro-optimized adjacent-only: 0


,time_value,publicID,text,depth_value,magnitude_value,magnitude_type,quality_standardError,depth_uncertainty,latitude_uncertainty,longitude_uncertainty,quality_associatedPhaseCount,creationInfo_author,event_type,creationInfo_agencyID,Observations


As you can see, the results are the same, but the time taken to run the function is significantly reduced. This is because we have avoided the overhead of pandas indexing in the loop, and we have also avoided the repeated concatenation of dataframes in each iteration. The time complexity of this function is still O(n), since we are only comparing each event with the next one in the list, but the constant factors have been reduced, which can make a big difference when there are many events in the dataset.

**There are a significant improvement of 95% in the time taken to run the function.** This micro-optimized version of the adjacent-only logic can be used as a baseline for further optimizations, such as using a more efficient algorithm that can reduce the time complexity of the search even further.

## Sorted Sliding Window Approach

Now, let's use a improved version created using the Sorted Sliding Window approach. We will also compare the results with the previous version of the function, which was based on a nested loop that compared all events with each other, and see how much time we can save with the new algorithm.

In [53]:
def check_duplicates_2(events: pd.DataFrame) -> pd.DataFrame:
    """
        Identifies duplicate events based on time and geographical distance. Optimized using a sorted sliding time window (O(n log n)).

    Parameters:
    -----------
    events : pd.DataFrame
        Seismic data for the given time range.

    Returns:
    --------
    duplicates : pd.DataFrame
        Table with information about duplicate events.
    """
    checks = ["earthquake", "volcanic eruption", "explosion", "outside of network interest"]
    selections = (
        events[events['event_type'].isin(checks)]
        .sort_values('time_value')
        .reset_index(drop=True)
    )

    # Edge case: empty or single-event dataset
    if len(selections) < 2:
        return pd.DataFrame(columns=columns + ['Observations'])

    # Pre-extract arrays for fast access (avoids per-row iloc overhead)
    times = selections['time_value'].values
    lats  = selections['latitude_value'].values
    lons  = selections['longitude_value'].values
    ids   = selections['publicID'].values

    TIME_WINDOW   = 4    # seconds
    DIST_THRESHOLD = 100  # km

    # Track which (i, j) pairs were already flagged to avoid duplicate rows
    flagged_pairs = set()
    dup_rows = []

    left = 0  # Left pointer of the sliding window

    for right in trange(1, len(selections), desc="Checking duplicates", unit=" events", leave=False):
        # Advance left pointer: drop events outside the 4-second window
        while left < right:
            delta = (times[right] - times[left]) / pd.Timedelta('1s')
            if delta > TIME_WINDOW:
                left += 1
            else:
                break

        # Compare `right` against every event still inside the window [left, right-1]
        for i in range(left, right):
            pair = (i, right) if i < right else (right, i)
            if pair in flagged_pairs:
                continue

            time_diff = abs(
                (selections.iloc[right]['time_value'] - selections.iloc[i]['time_value'])
                .total_seconds()
            )
            if time_diff > TIME_WINDOW:
                continue

            distance = hs.haversine((lats[i], lons[i]), (lats[right], lons[right]))
            if distance <= DIST_THRESHOLD:
                flagged_pairs.add(pair)

                row_i = selections.iloc[i][columns].copy()
                row_i['Observations'] = f'Possible duplicate event of {ids[right]}'

                row_j = selections.iloc[right][columns].copy()
                row_j['Observations'] = f'Possible duplicate event of {ids[i]}'

                dup_rows.extend([row_i, row_j])

    # Build duplicates dataframe in one shot (avoids repeated pd.concat overhead)
    if dup_rows:
        duplicates = pd.DataFrame(dup_rows, columns=columns + ['Observations']).reset_index(drop=True)
    else:
        duplicates = pd.DataFrame(columns=columns + ['Observations'])

    return duplicates

In [56]:
time1 = time.time()
duplicates_sstw = check_duplicates_2(event_df3)
time2 = time.time()
print(f"Number of duplicates found: {len(duplicates_sstw)}")
print(f"Time taken to run the function: {time2 - time1} seconds")

Number of duplicates found: 96
Time taken to run the function: 1.948444128036499 seconds


As you can see from here, the time taken to run the function is slightly higher than the micro-optimized adjacent-only version (51% slower), but it is still significantly lower than the original nested loop version (90% faster). Moreover, this method can find duplicates that are not adjacent in the list, as long as they are within the 4-second time window. This can be useful in cases where there are multiple events that occur within a short time frame, but are not necessarily adjacent in the list.

The time complexity of this function is O(n log n) due to the initial sorting step, and O(n * k) for the sliding window comparisons, where k is the average number of events within the 4-second window. In practice, this can be much faster than the O(n^2) complexity of the original nested loop approach, especially when there are many events in the dataset. Let's check if the duplicates found with this new approach are the same as the ones found with the adjacent-only logic, and see if there are any additional duplicates that were not found with the previous method.

In [64]:
# Check if the results are the same
print(f"Number of duplicates found with adjacent-only: {len(duplicates_adjacent)}")
print(f"Number of duplicates found with sorted sliding window: {len(duplicates_sstw)}")
differences = pd.concat([duplicates_sstw, duplicates_adjacent, duplicates_adjacent]).drop_duplicates(keep=False)
print(f"Number of additional duplicates found with sorted sliding window: {len(differences)}")
differences

Number of duplicates found with adjacent-only: 96
Number of duplicates found with sorted sliding window: 96
Number of additional duplicates found with sorted sliding window: 0


,time_value,publicID,text,depth_value,magnitude_value,magnitude_type,quality_standardError,depth_uncertainty,latitude_uncertainty,longitude_uncertainty,quality_associatedPhaseCount,creationInfo_author,event_type,creationInfo_agencyID,Observations
